# Vá chống quá khớp — Tầng 1

Notebook **tự chứa**, không phụ thuộc notebook nào khác. Chạy từ trên xuống. Không cần GPU.

| Ô | Việc |
|---|---|
| 0 | Thiết lập đường dẫn |
| 1 | **Vá augmentation** — thay pipeline chỉ-ColorJitter-0.1 bằng augmentation nhắm vào danh tính |
| 2 | Đọc `train_contrastive.py` để viết patch scheduler (chỉ đọc, không sửa) |
| 3 | Hoàn tác, nếu cần |

## Vì sao cần vá

Pipeline train hiện tại:

```
Resize(256) → CenterCrop(224) → ColorJitter(0.1, 0.1, 0.1, 0) → ToTensor → Normalize
```

`image_train` được gọi với `flip=0, offset_x=0, offset_y=0` hardcode, còn `PlaceCrop` và `SetFlip` thì bị comment trong `utils.py`. Nên mỗi ảnh được trình bày gần như y hệt nhau suốt 20 epoch — không có gì ngăn mạng học thuộc 72.674 ảnh cố định. Đó là lý do `train loss → 2e-05`.

Quá khớp ở đây là **ghi nhớ danh tính**: chỉ 15 khuôn mặt trong tập `fit`, mỗi người một video 4 phút liên tục. Nên augmentation phải nhắm vào tông da, ánh sáng, hình học khuôn mặt — không phải chỉ xê dịch khung hình.

## 0. Thiết lập

In [ ]:
from pathlib import Path
import shutil, datetime, ast, re, os

try:
    from google.colab import drive
    if not Path('/content/drive').exists(): drive.mount('/content/drive')
except ImportError: pass

SYM  = Path('/content/drive/MyDrive/Sym')
REPO = SYM/'Contrastive_V1'
ED   = REPO/'experiment_data.py'
TC   = REPO/'train_contrastive.py'

for n,p in [('REPO',REPO),('experiment_data.py',ED),('train_contrastive.py',TC)]:
    print(f'{"OK   " if p.exists() else "THIẾU"}  {n:24s} {p}')
assert ED.is_file() and TC.is_file(), 'Sai đường dẫn — sửa SYM rồi chạy lại'
print('\nSẵn sàng.')

## 1. Vá augmentation

An toàn ba lớp: sao lưu `.bak_<thời gian>` trước khi đụng; kiểm neo chỉ xuất hiện đúng một lần; `ast.parse` trước khi ghi. Sai bất kỳ bước nào thì dừng, không sửa file.

Chạy lại lần hai tự bỏ qua. `image_test` **giữ nguyên** — val và test không bao giờ được augment.

In [ ]:
AUG = '\nimport torchvision.transforms as _T\nclass strong_train_aug(object):\n    """Augmentation chong ghi nho danh tinh. Chu ky (img, *_) de thay image_train."""\n    def __init__(self, img_size=256, crop_size=224):\n        self.t = _T.Compose([\n            _T.Resize(img_size),\n            _T.RandomResizedCrop(crop_size, scale=(0.70, 1.0), ratio=(0.85, 1.18)),\n            _T.RandomHorizontalFlip(0.5),\n            _T.ColorJitter(0.4, 0.4, 0.4, 0.08),\n            _T.RandomGrayscale(0.2),\n            _T.RandomApply([_T.GaussianBlur(5, (0.1, 2.0))], 0.3),\n            _T.ToTensor(),\n            _T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),\n            _T.RandomErasing(0.25, scale=(0.02, 0.15)),\n        ])\n    def __call__(self, img, *a):\n        return self.t(img)\n'

ANCHOR = 'from torch.utils.data import Dataset, DataLoader'
OLD    = "self.transform = image_train(crop_size=crop_size) if split == 'fit'"
NEW    = "self.transform = strong_train_aug(crop_size=crop_size) if split == 'fit'"

src = ED.read_text(encoding='utf-8')

if 'strong_train_aug' in src:
    print('Đã vá trước đó — không làm gì.')
else:
    assert src.count(ANCHOR) == 1, f'neo import xuất hiện {src.count(ANCHOR)} lần — DỪNG'
    assert src.count(OLD)    == 1, f'dòng transform xuất hiện {src.count(OLD)} lần — DỪNG'
    stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    bak = REPO/f'experiment_data.py.bak_{stamp}'
    shutil.copy(ED, bak)
    new = src.replace(ANCHOR, ANCHOR + '\n' + AUG, 1).replace(OLD, NEW, 1)
    ast.parse(new)
    ED.write_text(new, encoding='utf-8')
    print(f'Sao lưu  : {bak.name}')
    print('Đã vá xong.')

chk = ED.read_text(encoding='utf-8')
print('\n--- kiểm lại ---')
print('class strong_train_aug  :', 'class strong_train_aug' in chk)
print('dùng cho split fit      :', NEW in chk)
print('image_test giữ nguyên   :', 'image_test(crop_size=crop_size)' in chk)
ast.parse(chk); print('cú pháp                 : OK')
print('\n--- các dòng liên quan ---')
for l in chk.splitlines():
    if 'strong_train_aug' in l or 'image_test(' in l or 'self.transform(img' in l:
        print('   ', l.strip()[:100])

## 2. Đọc để viết patch scheduler

**Chỉ đọc, không sửa gì.** Gửi lại toàn bộ output ô này.

Hiện `train_contrastive.py` không có scheduler nào — AdamW chạy `lr 1e-4` cố định suốt 20 epoch. Cần biết vòng lặp epoch nằm đâu để chèn `sched.step()` đúng chỗ.

In [ ]:
tc = TC.read_text(encoding='utf-8')

print('='*72); print('A. Tạo optimizer'); print('='*72)
i = tc.find('torch.optim.AdamW')
print(tc[max(0, i-500): i+600] if i >= 0 else '>>> KHÔNG thấy AdamW')

print('\n' + '='*72); print('B. Vòng lặp epoch'); print('='*72)
m = re.search(r'\n([ \t]*)for epoch in .*?(?=\n(?:def |class )|\Z)', tc, re.S)
print(m.group(0)[:3500] if m else '>>> KHÔNG thấy vòng for epoch')

print('\n' + '='*72); print('C. Danh sách hàm'); print('='*72)
for mm in re.finditer(r'\ndef (\w+)\(([^)]*)\)', tc):
    print(f'  def {mm.group(1)}({mm.group(2)[:72]})')

print('\n' + '='*72); print('D. Nơi ghi history / chọn best'); print('='*72)
for kw in ['history', 'best', 'selected_epoch', 'save', 'state_dict']:
    hits = [l.strip() for l in tc.splitlines() if kw in l]
    print(f'  {kw:16s} {len(hits):>2} dòng')
    for l in hits[:4]: print(f'        {l[:105]}')

## 3. Hoàn tác

Chỉ chạy nếu cần quay lại bản gốc. Nó lấy file `.bak_*` mới nhất.

In [ ]:
baks = sorted(REPO.glob('experiment_data.py.bak_*'))
if not baks:
    print('Không có bản sao lưu nào.')
else:
    print('Các bản sao lưu:')
    for b in baks: print('   ', b.name)
    # BỎ COMMENT hai dòng dưới để thực sự hoàn tác
    # shutil.copy(baks[-1], ED)
    # print(f'\nĐã khôi phục từ {baks[-1].name}')
    print('\n(đang ở chế độ chỉ xem — bỏ comment hai dòng trong ô này để hoàn tác thật)')

## Sau khi chạy xong

Gửi lại output **ô 2**. Tôi sẽ viết patch scheduler và lệnh train thử một fold với `--epochs 12 --batch-size 128 --workers 8`.

**Lưu ý:** sau khi vá, 9 checkpoint seed 0 hiện có **không còn so sánh được** với run mới — khác augmentation là khác điều kiện thí nghiệm. Chúng vẫn giữ giá trị làm bằng chứng cho chương chẩn đoán (`train loss 2e-05`, val đỉnh epoch 1, `AU2 F1 = 0` trên mặt lạ), nhưng bảng kết quả cuối phải train lại toàn bộ bằng cấu hình mới.